# ROGII — Wellbore Geology Prediction (Improved)

**Author:** Md Ashraf  
**Institute:** IIT (ISM) Dhanbad  
**Metric:** RMSE  
**Target:** TVT (True Vertical Thickness)  

---

### Key Improvements over Baseline
1. **Typewell KNN features** — match horizontal-well GR to typewell GR → lookup typewell TVT (geological template)
2. **GPU-enabled ensemble** — LightGBM + XGBoost + CatBoost (auto-detects GPU)
3. **Richer feature engineering** — depth normalization, NaN-zone distance, rolling min/max/quantiles, MD ratios
4. **Post-processing** — rows where `TVT_input` is known get exact values (no model needed)
5. **GroupKFold** by WELL — no leakage

## 1. Imports

In [3]:
import os
import gc
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ── GPU detection ─────────────────────────────────────────────────────────
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, timeout=10)
    GPU_AVAILABLE = result.returncode == 0
except Exception:
    GPU_AVAILABLE = False

print(f"GPU available : {GPU_AVAILABLE}")
print(f"pandas        : {pd.__version__}")
print(f"numpy         : {np.__version__}")
print(f"lightgbm      : {lgb.__version__}")
print(f"xgboost       : {xgb.__version__}")

GPU available : False
pandas        : 2.2.1
numpy         : 1.26.4
lightgbm      : 4.6.0
xgboost       : 3.2.0


## 2. Paths & Configuration

In [4]:
# ── Paths ─────────────────────────────────────────────────────────────────
# LOCAL
ROOT_DIR = Path("C:\\Users\\MD ASHRAF\\Documents\\wellbore-geology-prediction-Well-log\\data\\raw")
# KAGGLE
# ROOT_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/")

TRAIN_DIR       = ROOT_DIR / "train"
TEST_DIR        = ROOT_DIR / "test"
SUBMISSION_PATH = ROOT_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Column groups ─────────────────────────────────────────────────────────
TRAIN_ONLY_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
EXCLUDE_COLS    = ["WELL", "TVT", "id"] + TRAIN_ONLY_COLS

# ── Training config ───────────────────────────────────────────────────────
N_FOLDS     = 5
RANDOM_SEED = 42

# Ensemble weights: LightGBM, XGBoost, CatBoost
ENSEMBLE_WEIGHTS = [0.40, 0.30, 0.30]

print("Config OK")
print(f"  GPU     : {GPU_AVAILABLE}")
print(f"  N_FOLDS : {N_FOLDS}")

Config OK
  GPU     : False
  N_FOLDS : 5


## 3. Helper Functions

In [5]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def load_horizontal_wells(directory: Path, split: str) -> pd.DataFrame:
    """Load all horizontal well CSV files, add WELL / ROW_IDX / id columns."""
    files = sorted(directory.glob("*__horizontal_well.csv"))
    if not files:
        raise FileNotFoundError(f"No horizontal well files in {directory}")

    frames = []
    for f in tqdm(files, desc=f"Loading {split}"):
        well_name = f.stem.split("__")[0]
        df = pd.read_csv(f)
        df["WELL"]    = well_name
        df["ROW_IDX"] = np.arange(len(df))
        df["id"]      = well_name + "_" + df["ROW_IDX"].astype(str)
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    print(f"{split} shape : {out.shape}  |  wells : {out['WELL'].nunique()}")
    return out


def load_typewells(directory: Path) -> dict:
    """
    Load typewell CSVs — returns {well_name: DataFrame}.
    Typewell columns: TVT, GR, Geology (and possibly more).
    """
    typewell_files = sorted(directory.glob("*__typewell.csv"))
    typewells = {}
    for f in tqdm(typewell_files, desc="Loading typewells"):
        well_name = f.stem.split("__")[0]
        tw = pd.read_csv(f)
        tw = tw.dropna(subset=["GR", "TVT"])   # need both for KNN
        typewells[well_name] = tw
    print(f"Typewells loaded : {len(typewells)} wells")
    return typewells

## 4. Load Data

In [6]:
train_df = load_horizontal_wells(TRAIN_DIR, "TRAIN")
test_df  = load_horizontal_wells(TEST_DIR,  "TEST")

# Load typewells (available for train wells; may also exist for test wells)
typewells_train = load_typewells(TRAIN_DIR)
typewells_test  = load_typewells(TEST_DIR)
typewells_all   = {**typewells_train, **typewells_test}

# Sample submission
sample_sub = pd.read_csv(SUBMISSION_PATH)
print(f"\nSample submission : {sample_sub.shape}")

Loading TRAIN:   0%|          | 0/773 [00:00<?, ?it/s]

TRAIN shape : (5092255, 16)  |  wells : 773


Loading TEST:   0%|          | 0/3 [00:00<?, ?it/s]

TEST shape : (19221, 9)  |  wells : 3


Loading typewells:   0%|          | 0/773 [00:00<?, ?it/s]

Typewells loaded : 773 wells


Loading typewells:   0%|          | 0/3 [00:00<?, ?it/s]

Typewells loaded : 3 wells

Sample submission : (14151, 2)


## 5. Sanity Checks

In [7]:
print("── Train columns ──")
print(list(train_df.columns))

print("\n── TVT statistics ──")
print(train_df["TVT"].describe())

print(f"\nTVT_input NaN in train : {train_df['TVT_input'].isna().sum()}")
print(f"TVT_input NaN in test  : {test_df['TVT_input'].isna().sum()}")

# Verify submission IDs
sub_ids  = set(sample_sub["id"])
test_ids = set(test_df["id"])
matched  = sub_ids & test_ids
print(f"\nSub rows matched to test : {len(matched)} / {len(sub_ids)}")
assert len(matched) == len(sub_ids), "ID mismatch — review id construction"

── Train columns ──
['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input', 'WELL', 'ROW_IDX', 'id']

── TVT statistics ──
count    5.092255e+06
mean     1.150364e+04
std      6.399711e+02
min      9.245190e+03
25%      1.098793e+04
50%      1.135451e+04
75%      1.203826e+04
max      1.289389e+04
Name: TVT, dtype: float64

TVT_input NaN in train : 3783989
TVT_input NaN in test  : 14151

Sub rows matched to test : 14151 / 14151


## 6. Typewell Features

**Key idea:** The typewell provides a vertical reference log (GR vs TVT).  
By matching each horizontal-well GR value to the nearest typewell GR values (KNN),  
we retrieve a **typewell TVT estimate** — essentially a geological template lookup.

In [8]:
def add_typewell_features(df: pd.DataFrame, typewells: dict) -> pd.DataFrame:
    """
    For each well in df, fit a KNN regressor on typewell GR → TVT,
    then predict typewell TVT for each row based on its GR value.
    
    Features added:
      - tw_tvt_knn1   : best-match typewell TVT (k=1)
      - tw_tvt_knn5   : weighted-average typewell TVT (k=5)
      - tw_tvt_knn5_std : spread of top-5 typewell TVT matches (uncertainty)
      - tw_gr_residual : horizontal GR minus matched typewell GR (facies deviation)
    """
    df = df.copy()
    
    tw_tvt_knn1    = np.full(len(df), np.nan)
    tw_tvt_knn5    = np.full(len(df), np.nan)
    tw_tvt_knn5_std = np.full(len(df), np.nan)
    tw_gr_residual = np.full(len(df), np.nan)
    
    wells = df["WELL"].unique()
    
    for well in tqdm(wells, desc="Typewell features", leave=False):
        well_mask = df["WELL"] == well
        well_idx  = df.index[well_mask]
        
        tw = typewells.get(well)
        if tw is None or len(tw) < 5:
            # Fallback: use aggregate typewell from all available wells
            tw = pd.concat(list(typewells.values()), ignore_index=True)
            tw = tw.dropna(subset=["GR", "TVT"])
        
        # Feature matrix for KNN: GR — impute NaN with well median before KNN
        X_tw   = tw[["GR"]].values
        y_tw   = tw["TVT"].values

        gr_vals = df.loc[well_mask, "GR"].values.copy().astype(float)
        gr_median = np.nanmedian(gr_vals)
        if np.isnan(gr_median):
            gr_median = np.nanmedian(tw["GR"].values)  # fallback to typewell median
        gr_vals = np.where(np.isnan(gr_vals), gr_median, gr_vals)
        X_horz = gr_vals.reshape(-1, 1)

        # k=1: nearest typewell GR → TVT
        knn1 = KNeighborsRegressor(n_neighbors=1)
        knn1.fit(X_tw, y_tw)
        tw_tvt_knn1[well_mask.values] = knn1.predict(X_horz)
        
        # k=5: weighted average + std
        k = min(5, len(tw))
        knn5 = KNeighborsRegressor(n_neighbors=k, weights="distance")
        knn5.fit(X_tw, y_tw)
        tw_tvt_knn5[well_mask.values] = knn5.predict(X_horz)
        
        # Std of k=5 neighbors
        dists, idxs = knn5.kneighbors(X_horz)
        neighbor_tvts = y_tw[idxs]           # shape (n_rows, k)
        tw_tvt_knn5_std[well_mask.values] = neighbor_tvts.std(axis=1)
        
        # GR residual: horizontal GR minus best-match typewell GR
        nearest_gr = X_tw[idxs[:, 0], 0]
        tw_gr_residual[well_mask.values] = (
            df.loc[well_mask, "GR"].values - nearest_gr
        )
    
    df["tw_tvt_knn1"]    = tw_tvt_knn1
    df["tw_tvt_knn5"]    = tw_tvt_knn5
    df["tw_tvt_knn5_std"] = tw_tvt_knn5_std
    df["tw_gr_residual"] = tw_gr_residual
    
    return df


print("Adding typewell features to TRAIN...")
train_df = add_typewell_features(train_df, typewells_all)

print("Adding typewell features to TEST...")
test_df = add_typewell_features(test_df, typewells_all)

print("✓ Typewell features added")

Adding typewell features to TRAIN...


Typewell features:   0%|          | 0/773 [00:00<?, ?it/s]

Adding typewell features to TEST...


Typewell features:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Typewell features added


## 7. Feature Engineering

In [10]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Comprehensive feature engineering — all grouped by WELL to avoid cross-well leakage.
    """
    df = df.copy()

    # ── Rolling windows: GR & TVT_input ───────────────────────────────────
    windows = [3, 5, 10, 20, 50]

    for w in windows:
        grp_gr  = df.groupby("WELL")["GR"]
        grp_tvt = df.groupby("WELL")["TVT_input"]

        df[f"GR_roll_mean_{w}"]  = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f"GR_roll_std_{w}"]   = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).std())
        df[f"GR_roll_min_{w}"]   = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).min())
        df[f"GR_roll_max_{w}"]   = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).max())
        df[f"GR_roll_q25_{w}"]   = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).quantile(0.25))
        df[f"GR_roll_q75_{w}"]   = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).quantile(0.75))

        df[f"TVT_input_roll_mean_{w}"] = grp_tvt.transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f"TVT_input_roll_std_{w}"]  = grp_tvt.transform(lambda x: x.rolling(w, min_periods=1).std())

    # ── Lag features ──────────────────────────────────────────────────────
    lags = [1, 2, 3, 5, 10, 20]

    for lag in lags:
        df[f"GR_lag_{lag}"]        = df.groupby("WELL")["GR"].shift(lag)
        df[f"TVT_input_lag_{lag}"] = df.groupby("WELL")["TVT_input"].shift(lag)
        # Lead (future) features — valid because we have GR for entire well
        df[f"GR_lead_{lag}"]       = df.groupby("WELL")["GR"].shift(-lag)

    # ── Gradient (1st & 2nd derivative) ───────────────────────────────────
    df["GR_gradient"]         = df.groupby("WELL")["GR"].diff()
    df["TVT_input_gradient"]  = df.groupby("WELL")["TVT_input"].diff()
    df["GR_gradient2"]        = df.groupby("WELL")["GR_gradient"].diff()
    df["TVT_input_gradient2"] = df.groupby("WELL")["TVT_input_gradient"].diff()

    # ── Trajectory features ───────────────────────────────────────────────
    df["dX"] = df.groupby("WELL")["X"].diff()
    df["dY"] = df.groupby("WELL")["Y"].diff()
    df["dZ"] = df.groupby("WELL")["Z"].diff()

    df["traj_distance"] = np.sqrt(df["dX"]**2 + df["dY"]**2 + df["dZ"]**2)

    lateral = np.sqrt(df["dX"]**2 + df["dY"]**2).replace(0, np.nan)
    df["inclination_proxy"] = df["dZ"].abs() / lateral

    # Cumulative lateral distance (position along horizontal section)
    df["cum_lateral"] = df.groupby("WELL")["traj_distance"].cumsum()

    # ── Depth normalization (position within well) ─────────────────────────
    def normalize_md(g):
        mn, mx = g.min(), g.max()
        return (g - mn) / (mx - mn + 1e-9)

    df["MD_norm"] = df.groupby("WELL")["MD"].transform(normalize_md)

    # ── Distance from known TVT boundary (NaN zone start) ─────────────────
    # Per well: index of last non-NaN TVT_input
    def dist_to_nan_zone(g):
        known_mask = g.notna()
        if known_mask.all():        # fully observed well
            return pd.Series(np.zeros(len(g)), index=g.index)
        if not known_mask.any():    # fully unknown
            return pd.Series(np.arange(len(g), dtype=float), index=g.index)
        last_known = known_mask[::-1].idxmax()  # last True index
        pos = np.arange(len(g))
        last_pos = g.index.get_loc(last_known)
        dist = pos - last_pos
        dist[dist < 0] = 0   # positions before the boundary get 0
        return pd.Series(dist.astype(float), index=g.index)

    df["tvt_input_nan_dist"] = df.groupby("WELL")["TVT_input"].transform(
        dist_to_nan_zone
    )

    # ── TVT_input forward-fill (last known value propagated) ──────────────
    df["TVT_input_ffill"] = df.groupby("WELL")["TVT_input"].transform(
        lambda x: x.ffill()
    )

    # Difference between typewell prediction and ffill (model correction signal)
    df["tw_ffill_diff"] = df["tw_tvt_knn5"] - df["TVT_input_ffill"]

    # ── GR de-trended ─────────────────────────────────────────────────────
    df["GR_detrend_10"] = df["GR"] - df["GR_roll_mean_10"]
    df["GR_detrend_50"] = df["GR"] - df["GR_roll_mean_50"]

    # GR IQR (local facies spread)
    df["GR_iqr_10"] = df["GR_roll_q75_10"] - df["GR_roll_q25_10"]

    # ── Typewell TVT rolling stats (smooth out KNN noise) ─────────────────
    grp_tw = df.groupby("WELL")["tw_tvt_knn5"]
    df["tw_tvt_roll_mean_5"]  = grp_tw.transform(lambda x: x.rolling(5, min_periods=1).mean())
    df["tw_tvt_roll_mean_20"] = grp_tw.transform(lambda x: x.rolling(20, min_periods=1).mean())
    df["tw_tvt_gradient"]     = df.groupby("WELL")["tw_tvt_knn5"].diff()

    # ── Cumulative MD per well ─────────────────────────────────────────────
    df["MD_from_start"] = df.groupby("WELL")["MD"].transform(lambda x: x - x.iloc[0])

    # ── Fill NaNs from lags/diffs per-well ────────────────────────────────
    df = df.groupby("WELL", group_keys=False).apply(
        lambda g: g.ffill().bfill()
    )

    return df


print("Building features for TRAIN...")
train_df = build_features(train_df)

print("Building features for TEST...")
test_df  = build_features(test_df)

print(f"\nTrain shape after features : {train_df.shape}")
print(f"Test  shape after features : {test_df.shape}")

Building features for TRAIN...


MemoryError: Unable to allocate 660. MiB for an array with shape (17, 5092255) and data type float64

## 8. Build Feature Matrix

In [ ]:
common_cols = set(train_df.columns) & set(test_df.columns)

features = sorted(
    col for col in common_cols
    if col not in EXCLUDE_COLS
    and col != "ROW_IDX"
    and not col.startswith("_")
)

print(f"Total features : {len(features)}")
print(features)

# Verify no surface columns leaked
leaked = [c for c in TRAIN_ONLY_COLS if c in features]
assert not leaked, f"Surface columns leaked: {leaked}"

# Only rows with labelled TVT
train_mask = train_df["TVT"].notna()

X      = train_df.loc[train_mask, features].reset_index(drop=True)
y      = train_df.loc[train_mask, "TVT"].reset_index(drop=True)
groups = train_df.loc[train_mask, "WELL"].reset_index(drop=True)

X_test = test_df[features].reset_index(drop=True)

print(f"\nX      : {X.shape}")
print(f"y      : {y.shape}")
print(f"X_test : {X_test.shape}")
print(f"NaN in X      : {X.isna().sum().sum()}")
print(f"NaN in X_test : {X_test.isna().sum().sum()}")

## 9. Model Parameters

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────
lgb_params = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "boosting_type"    : "gbdt",
    "learning_rate"    : 0.02,
    "num_leaves"       : 127,
    "max_depth"        : -1,
    "feature_fraction" : 0.75,
    "bagging_fraction" : 0.75,
    "bagging_freq"     : 5,
    "min_child_samples": 15,
    "lambda_l1"        : 0.05,
    "lambda_l2"        : 0.05,
    "n_estimators"     : 8000,
    "random_state"     : RANDOM_SEED,
    "verbosity"        : -1,
    "n_jobs"           : -1,
    "device"           : "gpu" if GPU_AVAILABLE else "cpu",
}

# ── XGBoost ───────────────────────────────────────────────────────────────
xgb_params = {
    "objective"       : "reg:squarederror",
    "eval_metric"     : "rmse",
    "learning_rate"   : 0.02,
    "max_depth"       : 7,
    "subsample"       : 0.75,
    "colsample_bytree": 0.75,
    "min_child_weight": 5,
    "reg_alpha"       : 0.05,
    "reg_lambda"      : 1.0,
    "n_estimators"    : 8000,
    "random_state"    : RANDOM_SEED,
    "verbosity"       : 0,
    "n_jobs"          : -1,
    "tree_method"     : "hist",
    "device"          : "cuda" if GPU_AVAILABLE else "cpu",
}

# ── CatBoost ──────────────────────────────────────────────────────────────
cb_params = {
    "loss_function"   : "RMSE",
    "learning_rate"   : 0.02,
    "depth"           : 8,
    "l2_leaf_reg"     : 3.0,
    "iterations"      : 8000,
    "random_seed"     : RANDOM_SEED,
    "verbose"         : 0,
    "task_type"       : "GPU" if GPU_AVAILABLE else "CPU",
    "early_stopping_rounds": 200,
}

print(f"LGB device  : {lgb_params['device']}")
print(f"XGB device  : {xgb_params['device']}")
print(f"CatBoost    : {cb_params['task_type']}")

## 10. GroupKFold Training — LightGBM

In [ ]:
folds = GroupKFold(n_splits=N_FOLDS)

lgb_oof   = np.zeros(len(X))
lgb_test  = np.zeros(len(X_test))
lgb_scores = []
lgb_models = []

print(f"── LightGBM {N_FOLDS}-fold GroupKFold ──")

for fold, (tr_idx, val_idx) in enumerate(
    folds.split(X, y, groups), start=1
):
    print(f"\n  FOLD {fold}/{N_FOLDS}  "
          f"| train wells={groups.iloc[tr_idx].nunique()} "
          f"| val wells={groups.iloc[val_idx].nunique()}")

    X_tr, y_tr   = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=False),
            lgb.log_evaluation(period=1000)
        ]
    )

    val_preds = model.predict(X_val)
    lgb_oof[val_idx] = val_preds

    score = rmse(y_val, val_preds)
    lgb_scores.append(score)
    lgb_models.append(model)

    lgb_test += model.predict(X_test) / N_FOLDS
    print(f"  → LGB Fold RMSE : {score:.5f}")
    gc.collect()

print(f"\nLGB OOF RMSE : {rmse(y, lgb_oof):.5f}")
print(f"LGB CV mean  : {np.mean(lgb_scores):.5f} ± {np.std(lgb_scores):.5f}")

## 11. GroupKFold Training — XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_oof    = np.zeros(len(X))
xgb_test   = np.zeros(len(X_test))
xgb_scores = []

print(f"── XGBoost {N_FOLDS}-fold GroupKFold ──")

for fold, (tr_idx, val_idx) in enumerate(
    folds.split(X, y, groups), start=1
):
    print(f"\n  FOLD {fold}/{N_FOLDS}")

    X_tr, y_tr   = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = XGBRegressor(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=0,
        early_stopping_rounds=200
    )

    val_preds = model.predict(X_val)
    xgb_oof[val_idx] = val_preds

    score = rmse(y_val, val_preds)
    xgb_scores.append(score)

    xgb_test += model.predict(X_test) / N_FOLDS
    print(f"  → XGB Fold RMSE : {score:.5f}")
    gc.collect()

print(f"\nXGB OOF RMSE : {rmse(y, xgb_oof):.5f}")
print(f"XGB CV mean  : {np.mean(xgb_scores):.5f} ± {np.std(xgb_scores):.5f}")

## 12. GroupKFold Training — CatBoost

In [ ]:
cb_oof    = np.zeros(len(X))
cb_test   = np.zeros(len(X_test))
cb_scores = []

print(f"── CatBoost {N_FOLDS}-fold GroupKFold ──")

for fold, (tr_idx, val_idx) in enumerate(
    folds.split(X, y, groups), start=1
):
    print(f"\n  FOLD {fold}/{N_FOLDS}")

    X_tr, y_tr   = X.iloc[tr_idx].values, y.iloc[tr_idx].values
    X_val, y_val = X.iloc[val_idx].values, y.iloc[val_idx].values

    train_pool = Pool(X_tr, y_tr)
    val_pool   = Pool(X_val, y_val)

    model = CatBoostRegressor(**cb_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_preds = model.predict(X_val)
    cb_oof[val_idx] = val_preds

    score = rmse(y_val, val_preds)
    cb_scores.append(score)

    cb_test += model.predict(X_test.values) / N_FOLDS
    print(f"  → CB Fold RMSE : {score:.5f}")
    gc.collect()

print(f"\nCB OOF RMSE : {rmse(y, cb_oof):.5f}")
print(f"CB CV mean  : {np.mean(cb_scores):.5f} ± {np.std(cb_scores):.5f}")

## 13. Ensemble

In [ ]:
w_lgb, w_xgb, w_cb = ENSEMBLE_WEIGHTS

# ── Find optimal weights via OOF ─────────────────────────────────────────
from scipy.optimize import minimize

def ensemble_rmse(weights):
    w = np.array(weights)
    w = np.abs(w) / np.abs(w).sum()   # normalize to sum=1
    blend = w[0]*lgb_oof + w[1]*xgb_oof + w[2]*cb_oof
    return rmse(y, blend)

result = minimize(
    ensemble_rmse,
    x0=[0.40, 0.30, 0.30],
    method="Nelder-Mead",
    options={"maxiter": 1000, "xatol": 1e-6}
)

opt_w = np.abs(result.x) / np.abs(result.x).sum()
print(f"Optimized weights → LGB:{opt_w[0]:.3f} | XGB:{opt_w[1]:.3f} | CB:{opt_w[2]:.3f}")

# ── Final OOF blend ───────────────────────────────────────────────────────
oof_blend = opt_w[0]*lgb_oof + opt_w[1]*xgb_oof + opt_w[2]*cb_oof
print(f"\nEnsemble OOF RMSE (optimized) : {rmse(y, oof_blend):.5f}")
print(f"LGB OOF RMSE                  : {rmse(y, lgb_oof):.5f}")
print(f"XGB OOF RMSE                  : {rmse(y, xgb_oof):.5f}")
print(f"CB  OOF RMSE                  : {rmse(y, cb_oof):.5f}")

# ── Test predictions ──────────────────────────────────────────────────────
test_preds = opt_w[0]*lgb_test + opt_w[1]*xgb_test + opt_w[2]*cb_test

## 14. Post-processing

**Key insight:** Where `TVT_input` is **not NaN** in the test set, that value IS the TVT  
(same as in training). Overriding model predictions with exact known values can only help.

In [ ]:
# Build prediction dataframe
pred_df = test_df[["id", "TVT_input"]].copy().reset_index(drop=True)
pred_df["tvt_model"] = test_preds

# Override with known TVT_input where available
known_mask = pred_df["TVT_input"].notna()
pred_df["tvt"] = pred_df["tvt_model"]
pred_df.loc[known_mask, "tvt"] = pred_df.loc[known_mask, "TVT_input"]

print(f"Total test rows       : {len(pred_df)}")
print(f"Known (TVT_input set) : {known_mask.sum()}  → exact values used")
print(f"Unknown (model pred)  : {(~known_mask).sum()}  → model used")

# Evaluate on train OOF after the same post-processing
# (rows where TVT_input is not NaN in train = perfect prediction)
train_known = train_df.loc[train_mask, "TVT_input"].notna().values
oof_final = oof_blend.copy()
oof_final[train_known] = train_df.loc[train_mask, "TVT_input"].dropna().values

print(f"\nOOF RMSE (model only)      : {rmse(y, oof_blend):.5f}")
print(f"OOF RMSE (with post-proc)  : {rmse(y, oof_final):.5f}")

## 15. Feature Importance

In [ ]:
# LightGBM feature importance (gain-based across folds)
importance_matrix = np.column_stack(
    [m.booster_.feature_importance(importance_type="gain") for m in lgb_models]
)
mean_importance = importance_matrix.mean(axis=1)

importance_df = (
    pd.DataFrame({"feature": features, "importance": mean_importance})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 25 features (LightGBM gain):")
print(importance_df.head(25).to_string(index=False))

fig = px.bar(
    importance_df.head(30),
    x="importance", y="feature",
    orientation="h",
    title="Top 30 Features — LightGBM Gain (Mean across folds)",
    template="plotly_white"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 16. OOF Analysis

In [ ]:
oof_df = pd.DataFrame({
    "WELL"    : groups,
    "TVT_true": y.values,
    "TVT_pred": oof_blend,
})
oof_df["residual"] = oof_df["TVT_true"] - oof_df["TVT_pred"]

well_rmse = (
    oof_df.groupby("WELL")
    .apply(lambda g: rmse(g["TVT_true"], g["TVT_pred"]))
    .rename("RMSE")
    .sort_values(ascending=False)
    .reset_index()
)

print("Worst 10 wells by RMSE:")
print(well_rmse.head(10).to_string(index=False))

fig = px.histogram(
    oof_df, x="residual", nbins=100,
    title="OOF Residual Distribution — Ensemble",
    template="plotly_white"
)
fig.show()

## 17. Create Submission

In [ ]:
# Merge on id — order-independent
submission = sample_sub[["id"]].merge(
    pred_df[["id", "tvt"]], on="id", how="left"
)

n_missing = submission["tvt"].isna().sum()
if n_missing:
    print(f"⚠  {n_missing} NaN predictions → filling with train median")
    submission["tvt"] = submission["tvt"].fillna(y.median())
else:
    print("✓  All rows have valid predictions")

assert len(submission) == len(sample_sub), "Row count mismatch!"

out_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)

print(f"\nSubmission saved → {out_path}")
print(f"Shape : {submission.shape}")
print(submission.head(10))

## 18. Summary

In [ ]:
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  LGB  OOF RMSE          : {rmse(y, lgb_oof):.5f}")
print(f"  XGB  OOF RMSE          : {rmse(y, xgb_oof):.5f}")
print(f"  CB   OOF RMSE          : {rmse(y, cb_oof):.5f}")
print(f"  Ensemble OOF RMSE      : {rmse(y, oof_blend):.5f}")
print(f"  Ensemble + PostProc     : {rmse(y, oof_final):.5f}")
print(f"  Ensemble weights        : LGB={opt_w[0]:.3f} XGB={opt_w[1]:.3f} CB={opt_w[2]:.3f}")
print(f"  Features used           : {len(features)}")
print(f"  GPU                     : {GPU_AVAILABLE}")
print("=" * 60)